In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'

metadata_df

,sample_title,cell type
samples,,
GSM5688738,X1277013_BA.TPM,bladder
GSM5688739,X1375590_BA.TPM,bladder
GSM5688740,X1442434_BA.TPM,bladder
GSM5688741,X1443189_BA.TPM,bladder
GSM5688742,X1443723_BA.TPM,bladder
...,...,...
GSM5688811,tpm_CA1586932,bladder cancer cell
GSM5688812,tpm_CA1587363,bladder cancer cell
GSM5688813,tpm_CA1589480,bladder cancer cell


### Metadata from GEO is unnecessary here. All the information is provided in the supplementary data.

In [2]:
# supplementary data located here: https://pmc.ncbi.nlm.nih.gov/articles/instance/8718409/bin/Table_1.xlsx

expression_data_df = pd.read_excel('Table_1.xlsx', skiprows=1, sheet_name=2)
expression_data_df = expression_data_df.rename(columns={'ID': 'gene_symbol'})
expression_data_df = expression_data_df.set_index('gene_symbol')
expression_data_df = expression_data_df.T
expression_data_df.index.name = 'sample_title'


In [3]:
clinical_data_df = pd.read_excel('Table_1.xlsx', skiprows=1, sheet_name=1)
clinical_data_df = clinical_data_df.rename(columns={'Patient.number': 'sample_title'})
clinical_data_df = clinical_data_df.set_index('sample_title')

# clinical_data_df['Overall survival time'] = clinical_data_df['Overall survival time'].str.replace('survival day:', '', regex=False).str.strip()
# clinical_data_df['Overall survival status'] = clinical_data_df['Overall survival status'].str.replace('overall survival: ', '', regex=False).str.strip()

df_combined = clinical_data_df.merge(
    expression_data_df,
    left_on='sample_title',
    right_on='sample_title',
    how='inner'
)

# Drop accidental date-like feature columns introduced from source headers.
df_combined = df_combined.loc[:, pd.to_datetime(df_combined.columns, errors='coerce').isna()]


data_file_name = f'{GEO_ID}_original.csv'
df_combined.to_csv(data_file_name, index=True)
df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_2302/3742391447.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_combined = df_combined.loc[:, pd.to_datetime(df_combined.columns, errors='coerce').isna()]


,Age,Sex,Overall survival time,Overall survival status,Grade,Pathological Stage,Muscular_infiltration,FLT3LG,ERI1,SAP130,...,SLC10A3,TMEM141,ASPN,ZAN,RAB6A,PDPR,OR5B12,ST3GAL4,ZNF428,HIGD1A
sample_title,,,,,,,,,,,,,,,,,,,,,
tpm_CA1277013,58,Female,survival day: 377,overall survival: death,High,T4N0M0,MIBC,2.23,5.45,11.33,...,20.75,133.33,2.73,0.13,57.89,7.85,0.00,23.39,24.84,33.17
tpm_CA1308866,50,male,survival day:1518,overall survival: survival,High,T2N1M0,MIBC,17.90,5.56,21.74,...,70.66,93.95,4.14,0.00,104.52,13.68,0.00,105.12,29.61,92.48
tpm_CA1315330,63,male,survival day:946,overall survival: survival,High,T1NOMO,NMIBC,6.16,17.38,18.86,...,43.94,49.03,0.52,0.00,91.05,4.58,0.00,362.63,31.69,85.21
tpm_CA1366278,53,male,survival day:1258,overall survival: survival,High,T1NOMO,NMIBC,27.50,4.57,11.14,...,43.40,63.97,0.78,0.00,62.86,7.81,0.00,167.51,29.94,53.33
tpm_CA1375590,46,male,survival day:1213,overall survival: survival,High,T3bN1M0,MIBC,13.15,3.98,19.92,...,16.72,162.90,33.68,0.00,105.01,10.13,0.00,24.21,74.77,102.17
tpm_CA1382793,72,Female,survival day:1192,overall survival: survival,Low,T1N0M0,NMIBC,16.06,5.52,11.35,...,42.14,124.46,0.73,0.00,59.53,12.04,0.00,253.64,34.04,51.78
tpm_CA1409255,69,Female,survival day:1047,overall survival: survival,High,T3NOMO,MIBC,16.59,4.12,11.41,...,49.43,63.27,0.63,0.00,58.39,11.01,0.00,16.79,37.75,57.68
tpm_CA1427893,60,male,survival day:970,overall survival: survival,High,T1NOMO,NMIBC,15.62,1.85,6.77,...,41.87,82.72,0.94,0.00,60.66,2.63,0.00,209.48,12.39,90.64
tpm_CA1430225,57,male,survival day: 443,overall survival: death,High,T3NOMO,MIBC,13.44,2.19,11.45,...,57.70,121.90,0.46,0.00,76.63,7.37,0.00,50.25,22.14,99.70


In [4]:
'2021-09-06 00:00:00' in df_combined.columns

False

In [5]:
clinical_data_df.to_csv(f'{GEO_ID}_clinical_data.csv', index=True)

# Generate description


In [6]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_supplement

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_supplement(GEO_ID, 'GSE188715_clinical_data.csv')
    print("Description generated")

desc_data['data_summary'] = {
     'samples': clinical_data_df.shape[0],
    'clinical_features': clinical_data_df.reset_index().shape[1],
    'genes': expression_data_df.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)


Description updated
